In [ ]:
# Async prediction: the 7_prediction_continuous.ipynb methods refit
# on the schedule-driven async runs (data/clean_runs_seq.json; agents fire
# ~Poisson(0.5), silent agents carry). GPT-4o-mini; raw + balanced accuracy.
import json
from pathlib import Path

import numpy as np
import pandas as pd

from utils import (S_PREV, FIELD, D_POS, D_NEG, Logistic, adam, sigmoid,
                   load_data, graph_classes, build_xy, two_stage_fit, drives,
                   discrete_rollout, fcba, raw_acc)

# Load the question banks and persona embeddings from the shared dataset loader
_, banks, P, _ = load_data()                     # banks + persona PCA only

# Read the cleaned asynchronous snapshots and infer which graphs appear in training, testing, or both.
runs = json.load(open(Path("..") / "data" / "clean_runs_seq.json"))
GCLASS = graph_classes(runs, banks)


MODEL = "gpt-4o-mini"
REGIMES = ["subjective", "objective"]
T = 8
print(len(runs), "async episodes")

480 async episodes


In [ ]:
# continuous fit: MLE of (eps, w), Bernoulli P(s=+1) = (1 + m)/2, full-batch
# Adam (utils.adam), sigmoid-reparameterized eps; small L2 on the field weights
# only, on the raw design.
def fit_continuous(X, s_prev, y01, k, offset=0.0, l2=1e-3, iters=1500):
    """Returns (eps, w) for m = (1 - eps) s_prev + eps tanh(X w + offset);
    the last k columns of X are the couplings (unpenalized, init 0.1)."""
    N, dX = X.shape

    # Map the free clock parameter through a sigmoid, compute the tanh response, 
    # blend it with the previous state, and convert its mean to P(next state=+1).
    def grad_fn(theta):
        eps, w = sigmoid(theta[0]), theta[1:]
        h = np.tanh(X @ w + offset)
        m = (1.0 - eps) * s_prev + eps * h
        p = 0.5 * (1.0 + np.clip(m, -1.0 + 1e-7, 1.0 - 1e-7))

        # Penalize persona/question field terms, excluding the intercept and couplings
        # Differentiate the clipped Bernoulli loss for Adam.
        w_field = w[1:dX - k]                        # penalized: field sans bias
        nll = (-(y01 * np.log(p) + (1 - y01) * np.log(1 - p)).mean()
               + 0.5 * l2 * float(w_field @ w_field))
        inside = (np.abs(m) < 1.0 - 1e-7).astype(float)
        dL_dm = inside * (p - y01) / (2 * p * (1 - p)) / N
        g = np.zeros_like(theta)
        g[0] = float(np.sum(dL_dm * (h - s_prev))) * eps * (1 - eps)
        g[1:] = X.T @ (dL_dm * eps * (1 - h * h))
        g[2:dX - k + 1] += l2 * w_field
        return nll, g

    # Initialize couplings at 0.1 and other parameters at zero
    theta0 = np.zeros(1 + dX)
    theta0[1 + dX - k:] = 0.1                        # coupling init
    theta = adam(grad_fn, theta0, iters=iters)
    return float(sigmoid(theta[0])), theta[1:]

def continuous_rollout(phi, J_e, s0, eps, w, k, T=8):
    """Free-run carrying the continuous moment m;
    the reported spin at each step is sign(m)."""
    state = s0.astype(float)
    out = np.empty((len(s0), T, s0.shape[1]), dtype=int)
    for t in range(T):
        X = np.concatenate([phi, drives(J_e, state, k)], axis=-1)
        state = (1.0 - eps) * state + eps * np.tanh(X @ w)
        out[:, t] = np.where(state >= 0.0, 1, -1)
    return out

In [3]:
# the eight table methods; each returns (onestep, rollout) test predictions
def predict_all(x_tr, y_tr, x_te, J_te):
    rows = x_tr.reshape(-1, x_tr.shape[-1])          # pooled train transitions
    parsed = y_tr.reshape(-1) != 0                   # unparsed targets excluded
    y01 = (y_tr.reshape(-1)[parsed] > 0).astype(float)
    E, s0, phi = len(x_te), x_te[:, 0, :, S_PREV], x_te[:, 0, :, FIELD]
    pos, neg = rows[:, D_POS], rows[:, D_NEG]
    te_pos, te_neg = x_te[..., D_POS], x_te[..., D_NEG]
    out = {}

    # majority class: the train-majority spin, everywhere
    c = 1 if y01.mean() >= 0.5 else -1
    const = np.full((E, T, 32), c, dtype=int)
    out["Majority Class"] = (const, const)

    # persistence: no change; rollout frozen at s(0)
    out["Persistence"] = (x_te[..., S_PREV].astype(int),
                          np.repeat(s0[:, None].astype(int), T, axis=1))

    # interaction-free: logistic on the static field only (state-independent)
    clf = Logistic().fit(rows[parsed][:, FIELD], y01)
    pred = clf.predict_spin(x_te[..., FIELD])
    out["Interaction-Free"] = (pred, pred)

    # mean-field (Curie-Weiss): [field | population mean s̄(t)]
    def with_sbar(xx):
        sbar = xx[..., S_PREV].mean(axis=-1)
        return np.concatenate([xx[..., FIELD],
                               np.broadcast_to(sbar[..., None, None],
                                               xx.shape[:-1] + (1,))], axis=-1)
    clf = Logistic().fit(with_sbar(x_tr).reshape(-1, 17)[parsed], y01)
    onestep = clf.predict_spin(with_sbar(x_te))
    s, rollout = s0.copy(), np.empty((E, T, 32), dtype=int)
    for t in range(T):
        sbar = np.broadcast_to(s.mean(axis=1)[:, None, None], (E, 32, 1))
        s = clf.predict_spin(np.concatenate([phi, sbar], axis=-1)).astype(float)
        rollout[:, t] = s
    out["Mean-Field"] = (onestep, rollout)

    # discrete update, 1 coupling and (two-stage) 3 couplings
    clf = Logistic().fit(np.concatenate(
        [rows[:, FIELD], (pos + neg)[:, None]], axis=-1)[parsed], y01)
    onestep = clf.predict_spin(np.concatenate(
        [x_te[..., FIELD], (te_pos + te_neg)[..., None]], axis=-1))
    out["Discrete Update"] = (onestep, discrete_rollout(phi, J_te, s0, clf.w, 1))
    w3 = two_stage_fit(rows[:, FIELD], np.stack([pos, neg], axis=-1),
                       pos - neg, parsed, y01)
    d3_te = np.concatenate([x_te[..., FIELD], np.stack(
        [te_pos, te_neg, te_pos - te_neg], axis=-1)], axis=-1)
    out["Discrete + Three Couplings"] = (
        np.where(d3_te @ w3 > 0, 1, -1), discrete_rollout(phi, J_te, s0, w3, 3))

    # continuous update, 1 coupling: same design inside tanh, eps carry-over
    sp, y01f = rows[parsed][:, S_PREV], y01
    d1 = np.concatenate([rows[:, FIELD], (pos + neg)[:, None]], axis=-1)
    eps, w = fit_continuous(d1[parsed], sp, y01f, 1)
    m1 = (1 - eps) * x_te[..., S_PREV] + eps * np.tanh(np.concatenate(
        [x_te[..., FIELD], (te_pos + te_neg)[..., None]], axis=-1) @ w)
    out["Continuous Update"] = (np.where(m1 >= 0, 1, -1),
                                continuous_rollout(phi, J_te, s0, eps, w, 1))

    # continuous + 3 couplings, two-stage like the discrete: beta_0 on the
    # unsigned neighborhood first, then the signed betas with beta_0 fixed
    u = pos - neg
    _, w1 = fit_continuous(np.concatenate(
        [rows[:, FIELD], u[:, None]], axis=-1)[parsed], sp, y01f, 1)
    beta_0 = w1[16]
    eps, w2 = fit_continuous(np.concatenate(
        [rows[:, FIELD], np.stack([pos, neg], axis=-1)], axis=-1)[parsed],
        sp, y01f, 2, offset=(beta_0 * u)[parsed])
    w3c = np.concatenate([w2, [beta_0]])
    m1 = (1 - eps) * x_te[..., S_PREV] + eps * np.tanh(d3_te @ w3c)
    out["Continuous + Three Couplings"] = (
        np.where(m1 >= 0, 1, -1), continuous_rollout(phi, J_te, s0, eps, w3c, 3))
    return out

In [ ]:
# fit on train questions x J0-J7, score on test questions x seen (In-d.) vs
# fresh (OOD) graphs. Report raw accuracy and flip-and-class balanced accuracy.
METHOD_ROWS = ["Majority Class", "Persistence", "Interaction-Free",
               "Mean-Field", "Discrete Update", "Discrete + Three Couplings",
               "Continuous Update", "Continuous + Three Couplings"]
METRIC_ROWS = {"raw": METHOD_ROWS,
               "balanced": [m for m in METHOD_ROWS if m != "Majority Class"]}
METRIC_NAMES = {"raw": "raw accuracy",
                "balanced": "flip-and-class balanced accuracy"}
SPLITS = [("In-d.", "seen"), ("OOD", "fresh")]

results = {}
# Refit on asynchronous training questions and eight training graphs
# Evaluate the test questions on four seen and four fresh graphs.
for regime in REGIMES:
    x, y, ep = build_xy(MODEL, regime, runs, banks, P, GCLASS)
    train = (ep["split"] == "train") & np.isin(ep["gcls"], ["seen", "train_only"])
    test = (ep["split"] == "test") & np.isin(ep["gcls"], ["seen", "fresh"])
    preds = predict_all(x[train], y[train], x[test], ep["J"][test])
    y_te, s0, gc = y[test], x[test][:, 0, :, S_PREV], ep["gcls"][test]
    # Score raw and four-way balanced accuracy for both one-step and rollout predictions, preserving separate seen/fresh results.
    score = {"raw": lambda p, m: raw_acc(p, y_te, m),
             "balanced": lambda p, m: fcba(p, y_te, s0, m)}
    for method, (onestep, rollout) in preds.items():
        for metric, fn in score.items():
            results[metric, regime, method] = {
                split: (fn(onestep, gc == g), fn(rollout, gc == g))
                for split, g in SPLITS}
    print(f"{regime:10s}  train {train.sum()}, test {test.sum()}")

subjective  train 80, test 80


objective   train 160, test 160


In [5]:
# the tables in the async layout: one row per regime x graph split, one
# column per method; report both rollout and one-step predictions
def table(metric, kind):
    j = {"one-step": 0, "rollout": 1}[kind]
    return pd.DataFrame(
        {method: {f"{rg.capitalize()} {sp}":
                  round(results[metric, rg, method][sp][j], 1)
                  for rg in REGIMES for sp, _ in SPLITS}
         for method in METRIC_ROWS[metric]})

for metric in ("raw", "balanced"):
    for kind in ("rollout", "one-step"):
        print(f"=== {kind} — {METRIC_NAMES[metric]} ===")
        display(table(metric, kind))

=== rollout — raw accuracy ===


,Majority Class,Persistence,Interaction-Free,Mean-Field,Discrete Update,Discrete + Three Couplings,Continuous Update,Continuous + Three Couplings
Subjective In-d.,61.1,78.3,56.3,61.4,60.9,66.2,79.7,80.4
Subjective OOD,58.9,76.9,52.2,59.8,57.8,62.7,77.3,78.4
Objective In-d.,46.7,64.3,48.9,52.8,51.4,54.9,63.8,65.3
Objective OOD,48.2,65.0,48.0,55.1,53.0,54.7,64.7,65.9


=== one-step — raw accuracy ===


,Majority Class,Persistence,Interaction-Free,Mean-Field,Discrete Update,Discrete + Three Couplings,Continuous Update,Continuous + Three Couplings
Subjective In-d.,61.1,92.3,56.3,66.6,63.1,68.1,92.3,92.3
Subjective OOD,58.9,91.9,52.2,64.3,60.9,65.6,91.9,91.9
Objective In-d.,46.7,87.3,48.9,63.2,53.2,62.0,87.3,87.3
Objective OOD,48.2,87.6,48.0,64.4,53.9,63.7,87.6,87.6


=== rollout — flip-and-class balanced accuracy ===


,Persistence,Interaction-Free,Mean-Field,Discrete Update,Discrete + Three Couplings,Continuous Update,Continuous + Three Couplings
Subjective In-d.,58.0,52.3,55.2,55.3,59.4,57.4,58.6
Subjective OOD,56.2,48.8,52.5,53.5,57.0,55.6,56.3
Objective In-d.,49.1,49.9,50.8,50.2,53.1,49.5,51.1
Objective OOD,49.2,48.5,51.8,51.0,52.6,49.6,51.0


=== one-step — flip-and-class balanced accuracy ===


,Persistence,Interaction-Free,Mean-Field,Discrete Update,Discrete + Three Couplings,Continuous Update,Continuous + Three Couplings
Subjective In-d.,50.0,52.3,56.1,60.2,63.8,50.0,50.0
Subjective OOD,50.0,48.8,54.2,59.0,63.0,50.0,50.0
Objective In-d.,50.0,49.9,58.1,53.9,61.5,50.0,50.0
Objective OOD,50.0,48.5,59.2,53.7,62.5,50.0,50.0


In [6]:
# LaTeX tables of raw accuracy, with the best method per row in bold;
# export both rollout and one-step predictions
def latex(kind):
    j = {"one-step": 0, "rollout": 1}[kind]
    lines = [r"\begin{table}[h]", r"\centering", r"\small",
             r"\begin{tabular}{l cccc cc cc}", r"\toprule",
             r"& \multicolumn{4}{c}{Baselines}",
             r"& \multicolumn{2}{c}{Discrete Update}",
             r"& \multicolumn{2}{c}{Continuous Update} \\",
             r"\cmidrule(lr){2-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9}",
             r"& M & P & IF & MF", r"& Base & + Three Couplings",
             r"& Base & + Three Couplings \\", r"\midrule"]
    for rg in REGIMES:
        for sp, _ in SPLITS:
            vals = [results["raw", rg, m][sp][j] for m in METHOD_ROWS]
            best = max(vals)
            cs = [rf"\textbf{{{v:.1f}}}" if v == best else f"{v:.1f}"
                  for v in vals]
            lines.append(f"{rg.capitalize()} {sp:6s}& "
                         + " & ".join(cs) + r" \\")
    lines += [r"\bottomrule", r"\end{tabular}",
              rf"\caption{{\textit{{Prediction under asynchronous dynamics "
              rf"({kind}).}} Raw accuracy; GPT-4o-mini.}}",
              r"\end{table}"]
    return "\n".join(lines)

for kind in ("rollout", "one-step"):
    print(latex(kind) + "\n")

\begin{table}[h]
\centering
\small
\begin{tabular}{l cccc cc cc}
\toprule
& \multicolumn{4}{c}{Baselines}
& \multicolumn{2}{c}{Discrete Update}
& \multicolumn{2}{c}{Continuous Update} \\
\cmidrule(lr){2-5} \cmidrule(lr){6-7} \cmidrule(lr){8-9}
& M & P & IF & MF
& Base & + Three Couplings
& Base & + Three Couplings \\
\midrule
Subjective In-d. & 61.1 & 78.3 & 56.3 & 61.4 & 60.9 & 66.2 & 79.7 & \textbf{80.4} \\
Subjective OOD   & 58.9 & 76.9 & 52.2 & 59.8 & 57.8 & 62.7 & 77.3 & \textbf{78.4} \\
Objective In-d. & 46.7 & 64.3 & 48.9 & 52.8 & 51.4 & 54.9 & 63.8 & \textbf{65.3} \\
Objective OOD   & 48.2 & 65.0 & 48.0 & 55.1 & 53.0 & 54.7 & 64.7 & \textbf{65.9} \\
\bottomrule
\end{tabular}
\caption{\textit{Prediction under asynchronous dynamics (rollout).} Raw accuracy; GPT-4o-mini.}
\end{table}

\begin{table}[h]
\centering
\small
\begin{tabular}{l cccc cc cc}
\toprule
& \multicolumn{4}{c}{Baselines}
& \multicolumn{2}{c}{Discrete Update}
& \multicolumn{2}{c}{Continuous Update} \\
\cmidrule(lr